# Forecasting Readiness Workflow

This notebook builds a richer synthetic clinic operations dataset and then walks through time-series validation, leakage checks, and baseline evaluation.
The workflow is meant to show practical reasoning, not just function calls.

## Why generate a larger dataset?

The bundled sample CSV is intentionally tiny.
For forecasting readiness work, it is more useful to create a larger panel with multiple clinics and a longer history.

In [ ]:
from pathlib import Path
from pprint import pprint

import pandas as pd

from ds_workspace_mcp.diagnostics import detect_possible_target_leakage_dataset
from ds_workspace_mcp.ml.baselines import evaluate_baseline_model_dataset
from ds_workspace_mcp.synthetic.healthcare import GeneratorConfig, write_healthcare_dataset
from ds_workspace_mcp.timeseries import validate_time_series_dataset

In [ ]:
output_path = Path("data") / "notebook_clinic_usage.csv"
config = GeneratorConfig(
    output_path=output_path,
    start_date="2025-01-01",
    days=180,
    clinics=5,
    seed=42,
)

written_path = write_healthcare_dataset(config)
written_path

In [ ]:
df = pd.read_csv(output_path)
df.head()

The generated dataset includes a natural forecasting target: `appointments_completed`.
It also includes operational drivers such as staffing, wait time, campaigns, and holidays.

In [ ]:
validation = validate_time_series_dataset(
    file_name=output_path.name,
    time_column="date",
    target_column="appointments_completed",
    group_column="clinic_id",
)

pprint(validation.model_dump())

## Readiness interpretation

Key questions for the validation result:

- Is the date column parseable and consistently sorted within each clinic?
- Are there missing intervals that would complicate daily forecasting?
- Are there missing target values that require imputation or row filtering?

In [ ]:
leakage = detect_possible_target_leakage_dataset(
    file_name=output_path.name,
    target_column="appointments_completed",
)

leakage.warnings[:10]

Leakage warnings here are heuristic review signals, not proof of a modeling flaw.
Columns such as wait time or same-day staffing may still need a business-timeline check before being used in a true forecast.

In [ ]:
baseline = evaluate_baseline_model_dataset(
    file_name=output_path.name,
    target_column="appointments_completed",
    task_type="regression",
    test_size=0.2,
    random_state=42,
)

baseline.model_dump()

## Baseline caveat

The current baseline helper uses a standard random train/test split.
That is acceptable for a quick reference baseline, but it is not a proper forecasting backtest.
A production forecasting workflow should replace this with a time-aware split or rolling-origin evaluation.

## Suggested next modeling steps

1. Freeze a forecasting target definition and prediction horizon per clinic.
2. Remove or lag any features not known at prediction time.
3. Add calendar features such as weekday, month, and holiday proximity.
4. Replace the random split baseline with a chronological validation routine.
5. Compare naive, seasonal naive, and tree-based regression baselines.